In [3]:
import json
import os
import sys
from pathlib import Path
from typing import Any

import pydantic
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

from PydanticContracts import SyntheticChunkingExample

load_dotenv()

### Generator

MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
REASONING = True
REASONING_EFFORT = "medium"
MAX_TOKENS = (8192, 10000)[REASONING]
TIMEOUT_SECONDS = 240.0
PAIRS_PER_PROMPT = 3
REGENERATION_ATTEMPTS = 3

### Judge
JUDGE_MODEL_NAME = "deepseek-v4-pro"
JUDGE_BASE_URL = "https://api.deepseek.com"
JUDGE_TEMPERATURE = 0.8
JUDGE_REASONING = True
JUDGE_REASONING_EFFORT = "medium"
JUDGE_MAX_TOKENS = (8192, 10000)[REASONING]
JUDGE_TIMEOUT_SECONDS = 240.0
JUDGE_PAIRS_PER_PROMPT = 32
JUDGE_REGENERATION_ATTEMPTS = 3

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
SELECTED_PROMPTS = [
    Path("general_validation.md"),
    # Path("metrics/size_compliance.md"),
    Path("metrics/intrachunk_cohesion.md"),
    Path("metrics/contextual_coherence.md"),
    Path("metrics/boundary_clarity.md"),
    Path("metrics/chunk_score.md"),
    Path("metrics/hope_concept_unity.md"),
    Path("metrics/hope_semantic_independence.md"),
    Path("metrics/hope_information_preservation.md"),
]

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)

In [5]:
def save_json(data: dict[str, Any] | list[dict[str, Any]], path: Path) -> Path:
    """Save JSON objects in a human-readable UTF-8 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

In [7]:
SyntheticChunkingExample.model_json_schema()

{'$defs': {'ChunkingVariant': {'properties': {'chunks': {'items': {'type': 'string'},
     'title': 'Chunks',
     'type': 'array'},
    'rationale': {'title': 'Rationale', 'type': 'string'},
    'focus': {'additionalProperties': True,
     'title': 'Focus',
     'type': 'object'}},
   'required': ['chunks', 'rationale'],
   'title': 'ChunkingVariant',
   'type': 'object'}},
 'properties': {'document_title': {'title': 'Document Title',
   'type': 'string'},
  'source_document': {'title': 'Source Document', 'type': 'string'},
  'positive': {'$ref': '#/$defs/ChunkingVariant'},
  'negative': {'$ref': '#/$defs/ChunkingVariant'},
  'controlled_change': {'title': 'Controlled Change', 'type': 'string'},
  'expected_relation': {'const': 'positive_higher_than_negative',
   'title': 'Expected Relation',
   'type': 'string'}},
 'required': ['document_title',
  'source_document',
  'positive',
  'negative',
  'controlled_change',
  'expected_relation'],
 'title': 'SyntheticChunkingExample',
 'type

In [ ]:
def generate(
    system_prompt,
    user_prompt,
):
    result = None
    for attempt in range(REGENERATION_ATTEMPTS):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": ("disabled", "enabled")[REASONING]}},
            reasoning_effort=REASONING_EFFORT,
        )
        content = response.choices[0].message.content
        try:
            # ic(response)
            # ic(content)
            result = SyntheticChunkingExample.model_validate_json(content).model_dump()
            break
        except pydantic.ValidationError:
            tqdm.write("Retrying..")
    return result

In [ ]:
# def llm_judge(content):


In [ ]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")

for prompt_path in tqdm(SELECTED_PROMPTS, desc="Prompts", position=0):
    prompt_name = prompt_path.stem
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    results = []
    output_path = ""
    for pair_number in tqdm(
        range(1, PAIRS_PER_PROMPT + 1), desc="Items", position=1, leave=False
    ):
        result = generate(system_prompt=system_prompt, user_prompt=user_prompt)

        results.append(result)

        output_path = save_json(results, OUTPUT_ROOT / f"{prompt_name}.json")
    tqdm.write(f"Saved {output_path}")